In [1]:
import pandas as pd, numpy as np
pd.set_option("display.max_columns", None)

raw = pd.read_csv("../data/raw/autodesk_materials.csv")
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
print("\nMissing fraction per column:\n", raw.isna().mean().round(2))
print("\nSummary:\n", raw.describe().T)
print("\nStandards:\n", raw["Std"].value_counts().head(10))
print("\nSample of material names:")
print(raw["Material"].sample(40, random_state=0).tolist())

Shape: (1552, 15)
Columns: ['Std', 'ID', 'Material', 'Heat treatment', 'Su', 'Sy', 'A5', 'Bhn', 'E', 'G', 'mu', 'Ro', 'pH', 'Desc', 'HV']

Missing fraction per column:
 Std               0.00
ID                0.00
Material          0.00
Heat treatment    0.48
Su                0.00
Sy                0.00
A5                0.13
Bhn               0.70
E                 0.00
G                 0.00
mu                0.00
Ro                0.00
pH                0.88
Desc              0.37
HV                0.89
dtype: float64

Summary:
       count           mean            std      min       25%       50%  \
Su   1552.0     572.753222     326.834927     69.0     340.0     500.0   
A5   1346.0      19.329421      12.422004      0.5      11.0      16.0   
Bhn   463.0     177.138229     113.508044     19.0      78.0     170.0   
E    1552.0  164571.520619   56135.405867  73000.0  105000.0  206000.0   
G    1552.0   85598.840206  125326.800138  26000.0   40000.0   79000.0   
mu   1552.0     

In [2]:
# 1) Why is Sy not numeric?
print(raw.dtypes)
sy_num = pd.to_numeric(raw["Sy"], errors="coerce")
bad = raw["Sy"].notna() & sy_num.isna()
print("\nNon-numeric Sy cells:", bad.sum())
print(raw.loc[bad, ["Material", "Sy"]].head(15))

# 2) Hardness overlap
has_hv, has_bhn = raw["HV"].notna(), raw["Bhn"].notna()
print("\nHV only:", (has_hv & ~has_bhn).sum(),
      "| Bhn only:", (~has_hv & has_bhn).sum(),
      "| both:", (has_hv & has_bhn).sum(),
      "| neither:", (~has_hv & ~has_bhn).sum())

# 3) What kinds of materials? (first word of the name)
print("\n", raw["Material"].str.split().str[0].value_counts().head(25))

# 4) Anything turbine-like?
desc = raw["Desc"].astype(str)
for k in ["titan", "nickel", "inconel", "hastelloy", "stainless", "tool", "cast iron", "magnes", "alumin", "copper", "bronze", "brass"]:
    m = raw["Material"].str.contains(k, case=False, na=False) | desc.str.contains(k, case=False, na=False)
    print(f"{k:12s}{m.sum()}")

# 5) Near-duplicates
fp = ["Su", "Sy", "E", "Ro"]
print("\nRows sharing identical (Su,Sy,E,Ro) with another row:", raw.duplicated(subset=fp, keep=False).sum())
print("Unique property combinations:", raw.drop_duplicates(subset=fp).shape[0])

# 6) What does Desc contain?
print("\n", raw["Desc"].dropna().sample(15, random_state=1).tolist())

Std                   str
ID                    str
Material              str
Heat treatment        str
Su                  int64
Sy                    str
A5                float64
Bhn               float64
E                   int64
G                   int64
mu                float64
Ro                  int64
pH                float64
Desc                  str
HV                float64
dtype: object

Non-numeric Sy cells: 8
         Material       Sy
625      EN FeP01  280 max
626      EN FeP03  240 max
627      EN FeP04  210 max
716   DIN USt1303  250 max
717    DIN St1405  210 max
718    DIN St1203  280 max
1397        NF ZE  240 max
1398       NF ZES  225 max

HV only: 165 | Bhn only: 463 | both: 0 | neither: 924

 Material
Steel            454
Aluminum         251
CSN              172
DIN              114
EN                80
NF                80
Copper            77
BS                61
JIS               56
Magnesium         37
Grey              36
Nodular           28
Malleable 